In [1]:
!pip install pandas numpy scikit-learn joblib streamlit plotly pyngrok --quiet


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 49.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 77.5 MB/s eta 0:00:00


In [2]:
import os
os.makedirs("data", exist_ok=True)
os.makedirs("models", exist_ok=True)


In [6]:
#training the model
import pandas as pd, numpy as np, joblib, os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestRegressor

DATA_DIR="data"; MODEL_DIR="models"
os.makedirs(MODEL_DIR,exist_ok=True)
COUNTIES=["nakuru","kisumu","machakos","meru","turkana"]
FEATURES=["temperature","rainfall","soil_moisture","crop_type"]; TARGET="drought_index"

def prep(df):
    df=df.dropna(subset=FEATURES+[TARGET])
    for c in df.columns:
        if df[c].dtype=="object": df[c]=LabelEncoder().fit_transform(df[c])
    X=StandardScaler().fit_transform(df[FEATURES]); y=df[TARGET]
    return X,y

def train_one(c):
    path=f"{DATA_DIR}/{c}_weather.csv"
    if not os.path.exists(path): print("no data",c); return None
    df=pd.read_csv(path)
    X,y=prep(df)
    Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.2,random_state=42)
    m=RandomForestRegressor(n_estimators=100,random_state=42)
    m.fit(Xtr,ytr)
    print(f"{c}: R²={m.score(Xte,yte):.2f}")
    joblib.dump(m,f"{MODEL_DIR}/{c}_model.pkl")
    return m

def aggregate(ms):
    trees=[]
    for m in ms:
        if m: trees.extend(m.estimators_)
    G=RandomForestRegressor(n_estimators=len(trees))
    G.estimators_=trees
    joblib.dump(G,f"{MODEL_DIR}/global_model.pkl")
    print("🌍 global_model.pkl ready")

if __name__=="__main__":
    models=[train_one(c) for c in COUNTIES]
    aggregate([m for m in models if m])


nakuru: R²=0.95
kisumu: R²=0.95
machakos: R²=0.95
meru: R²=0.95
turkana: R²=0.94
🌍 global_model.pkl ready


In [7]:
!python train_dml.py


nakuru: R²=0.95
kisumu: R²=0.95
machakos: R²=0.95
meru: R²=0.95
turkana: R²=0.94
🌍 global_model.pkl ready


In [20]:
%%writefile drought_dashboard.py
import streamlit as st
import pandas as pd
import numpy as np
import os, joblib
from sklearn.preprocessing import LabelEncoder
import plotly.express as px

# --- Page setup
st.set_page_config(page_title="Kenya Drought Prediction", layout="wide")
st.markdown("## 🌾 Distributed Drought Prediction Dashboard — Kenyan Agricultural Sector")

# --- County selector
counties = ["Nakuru", "Kisumu", "Machakos", "Meru", "Turkana"]
county = st.sidebar.selectbox("Select County", counties)

data_path = f"data/{county.lower()}_weather.csv"
model_path = f"models/{county.lower()}_model.pkl"

# --- Load dataset
if not os.path.exists(data_path):
    st.error(f"No dataset found for {county}. Please check {data_path}.")
    st.stop()

df = pd.read_csv(data_path)
st.subheader(f"📊 Data Preview: {county}")
st.dataframe(df.head())

# --- Encode categorical columns
for c in df.columns:
    if df[c].dtype == "object":
        df[c] = LabelEncoder().fit_transform(df[c].astype(str))

# --- Load trained model
if os.path.exists(model_path):
    model = joblib.load(model_path)
else:
    model = joblib.load("models/global_model.pkl")

# ✅ --- USE ONLY THE 4 TRAINING FEATURES ---
FEATURE_COLS = ["temperature", "rainfall", "soil_moisture", "crop_type"]
missing_cols = [c for c in FEATURE_COLS if c not in df.columns]
if missing_cols:
    st.error(f"Missing expected columns in data: {missing_cols}")
    st.stop()

X = df[FEATURE_COLS].copy()

# --- Make predictions
pred = model.predict(X)
df["Drought_Risk_Index"] = np.clip(pred, 0, 1)
avg = df["Drought_Risk_Index"].mean()

# --- Display average drought risk
st.metric("Average Drought Risk Index", f"{avg:.2f}")
if avg < 0.3:
    st.success("✅ Low drought risk — favorable conditions.")
elif avg < 0.7:
    st.warning("⚠️ Moderate drought risk — monitor regularly.")
else:
    st.error("🚨 High drought risk — mitigation needed.")

# --- Plot risk over time
if "date" in df.columns:
    fig = px.line(df, x="date", y="Drought_Risk_Index",
                  title=f"{county} — Drought Risk Over Time", markers=True)
else:
    fig = px.line(df, y="Drought_Risk_Index",
                  title=f"{county} — Drought Risk Over Time", markers=True)
st.plotly_chart(fig, use_container_width=True)

# --- Optional rainfall vs soil moisture chart
if "rainfall" in df.columns and "soil_moisture" in df.columns:
    st.markdown("### 💧 Rainfall vs Soil Moisture")
    fig2 = px.scatter(df, x="rainfall", y="soil_moisture",
                      color="Drought_Risk_Index",
                      labels={"rainfall": "Rainfall (mm)", "soil_moisture": "Soil Moisture (%)"},
                      title=f"{county} — Rainfall vs Soil Moisture Impact")
    st.plotly_chart(fig2, use_container_width=True)

st.markdown("---")
st.markdown("📡 *Powered by Distributed Machine Learning for Kenya's Agricultural Sector*")


Overwriting drought_dashboard.py


In [22]:
from pyngrok import ngrok

ngrok.set_auth_token('3B3nVo8GmXxyZrdpdVGQW3TWthM_6bK4U9u1eRMqVLdkFdn2i')

!pkill streamlit || true
public_url = ngrok.connect(8501)
print("🔗 App URL:", public_url)
!streamlit run drought_dashboard.py --server.port 8501

🔗 App URL: NgrokTunnel: "https://unexempting-boyish-mercy.ngrok-free.dev" -> "http://localhost:8501"
Traceback (most recent call last):
  File "/usr/local/bin/streamlit", line 5, in <module>
    from streamlit.web.cli import main
  File "/usr/local/lib/python3.12/dist-packages/streamlit/__init__.py", line 62, in <module>
    from streamlit import config as _config
  File "/usr/local/lib/python3.12/dist-packages/streamlit/config.py", line 31, in <module>
    from streamlit import config_util, development, env_util, file_util, util
  File "/usr/local/lib/python3.12/dist-packages/streamlit/config_util.py", line 27, in <module>
    from streamlit import cli_util, url_util
  File "/usr/local/lib/python3.12/dist-packages/streamlit/cli_util.py", line 22, in <module>
    from streamlit import env_util, errors
  File "<frozen importlib._bootstrap>", line 1360, in _find_and_load
  File "<frozen importlib._bootstrap>", line 1322, in _find_and_load_unlocked
  File "<frozen importlib._bootstrap>", 